# Current example of splitting code with CV

In [2]:
from PreRun import PreRun, PostRun
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import root_mean_squared_error as rmse
from itertools import product
from datetime import date, datetime
from by_dates_Kfold import k_fold_split_option_a
from sklearn.linear_model import RidgeCV

### Adjust these paths to your current standard


In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
# I have current temp-Clean results in my working folder -- you may have them elsewhere!
inverter_4903_path = Path('./test_results/4903-inverter/')

In [4]:
shorter_test = PreRun(4903, inverter_4903_path, 'inverter', systems_cleaned)
shorter_test.add_weather_features_only()
shorter_test.add_energy_features_only(daily_lags=1, include_hour_cyclic=True, include_day_of_year_cyclic=True)
shorter_test.good_end_days_naive(streak=7)

,date
0,2014-09-11
1,2014-09-12
2,2014-09-13
3,2014-09-14
4,2014-09-15
...,...
422,2017-12-06
423,2017-12-07
424,2017-12-08
425,2017-12-22


In [5]:
am_dat_external = shorter_test.amended_data.copy(deep=True)
good_ends_external = shorter_test.end_days_naive

### add a year column

In [6]:
am_dat_external.loc[:, 'year'] = am_dat_external['time'].dt.year

In [7]:
am_dat_external.head()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year
0,2014-08-02 00:00:00,6.5,1.00,0.0,0.0,-0.501242,-0.865307,0.000000,1.000000,2014
1,2014-08-02 01:00:00,6.5,1.00,0.0,0.0,-0.501242,-0.865307,0.258819,0.965926,2014
2,2014-08-02 02:00:00,6.5,0.99,0.0,0.0,-0.501242,-0.865307,0.500000,0.866025,2014
3,2014-08-02 03:00:00,6.5,0.90,0.0,0.0,-0.501242,-0.865307,0.707107,0.707107,2014
4,2014-08-02 04:00:00,6.5,0.88,0.0,0.0,-0.501242,-0.865307,0.866025,0.500000,2014


#### Naive Train-test-split to keep things going

In [8]:
df_train = am_dat_external[am_dat_external['time'] < pd.Timestamp(datetime(2017, 1, 1))]
df_test = am_dat_external[am_dat_external['time'] >= pd.Timestamp(datetime(2017, 1, 1))]

In [9]:
# choose the first 50 good-ends (after trimming) for validation days
my_splits_front = k_fold_split_option_a(
    df_train=df_train,
    good_ends=good_ends_external,
    n_splits=50,
    window_size=None,
    gap_day=True,
    front_or_back='front',
    return_type='index'
)

#### Sample

In [10]:
my_splits_front[12]

(RangeIndex(start=0, stop=3866, step=1),
 RangeIndex(start=3879, stop=3893, step=1))

#### Aside -- the way I'm returning indices, use `loc` rather than `iloc` to access particular rows!

In [11]:
df_train.loc[3860:3865, ['time', 'energy']]

,time,energy
3860,2015-08-26 13:00:00,38.206167
3861,2015-08-26 14:00:00,44.925000
3862,2015-08-26 15:00:00,35.918583
3863,2015-08-26 16:00:00,24.172833
3864,2015-08-26 17:00:00,7.798117
3865,2015-08-26 18:00:00,0.283233


In [12]:
df_train.loc[3879:3892, ['time', 'energy']]

,time,energy
3879,2015-08-28 05:00:00,0.040800
3880,2015-08-28 06:00:00,5.028350
3881,2015-08-28 07:00:00,17.384333
3882,2015-08-28 08:00:00,31.715000
3883,2015-08-28 09:00:00,39.899833
3884,2015-08-28 10:00:00,48.688667
3885,2015-08-28 11:00:00,47.057000
3886,2015-08-28 12:00:00,37.459833
3887,2015-08-28 13:00:00,48.599833
3888,2015-08-28 14:00:00,34.966167


### Drop lags for testing purposes

In [13]:
my_cols = ['year', 'hour_cos', 'hour_sin', 'day_of_year_cos', 'day_of_year_sin', 'proportion_daytime', 'global_tilted_irradiance', 'cloud_cover']

In [14]:
X_train = df_train[my_cols]
y_train = df_train['energy']
X_test = df_test[my_cols]
y_test = df_test['energy']

### Apply to RidgeCV as a sample of CV input

In [15]:
test_cv = RidgeCV(alphas=(0.001, 0.01, 0.01, 1, 10, 100, 1000),
                  fit_intercept=True, scoring='neg_mean_squared_error',
                  cv=my_splits_front)  # results of the above

In [16]:
test_cv.fit(X_train, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","(0.001, ...)"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.","[(RangeIndex(st...=3535, step=1), ...), (RangeIndex(st...=3549, step=1), ...), ...]"
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [17]:
y_pred = test_cv.predict(X_test)

In [18]:
rmse(y_true=y_test, y_pred=y_pred)

7.982114547261204

#### What of other options?

In [19]:
# Take the back 50 splits (from dates in df_train),
# with a window size of 5
# and no gap day
my_splits_back = k_fold_split_option_a(
    df_train=df_train,
    good_ends=good_ends_external,
    n_splits=50,
    window_size=5,
    gap_day=False,
    front_or_back='back',
    return_type='DataFrame'
)

In [20]:
my_splits_back[-1][0].head()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year,date
8604,2016-12-10 07:00:00,0.421250,0.35,0.000000,0.701667,-0.368763,0.929523,0.965926,-0.258819,2016,2016-12-10
8605,2016-12-10 08:00:00,4.565683,0.22,35.001354,1.000000,-0.368763,0.929523,0.866025,-0.500000,2016,2016-12-10
8606,2016-12-10 09:00:00,23.690167,0.00,216.822723,1.000000,-0.368763,0.929523,0.707107,-0.707107,2016,2016-12-10
8607,2016-12-10 10:00:00,30.256700,0.01,401.899536,1.000000,-0.368763,0.929523,0.500000,-0.866025,2016,2016-12-10
8608,2016-12-10 11:00:00,35.793000,0.07,536.256714,1.000000,-0.368763,0.929523,0.258819,-0.965926,2016,2016-12-10


In [21]:
my_splits_back[-1][0].tail()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year,date
8647,2016-12-14 12:00:00,32.925833,0.11,578.392822,1.0000,-0.304115,0.952635,1.224647e-16,-1.000000,2016,2016-12-14
8648,2016-12-14 13:00:00,28.763333,0.24,573.289062,1.0000,-0.304115,0.952635,-2.588190e-01,-0.965926,2016,2016-12-14
8649,2016-12-14 14:00:00,11.575950,0.74,522.980774,1.0000,-0.304115,0.952635,-5.000000e-01,-0.866025,2016,2016-12-14
8650,2016-12-14 15:00:00,4.278183,1.00,336.990509,1.0000,-0.304115,0.952635,-7.071068e-01,-0.707107,2016,2016-12-14
8651,2016-12-14 16:00:00,0.040267,1.00,108.478371,0.7875,-0.304115,0.952635,-8.660254e-01,-0.500000,2016,2016-12-14


In [22]:
my_splits_back[-1][1].head()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year,date
8652,2016-12-15 07:00:00,0.097733,0.11,0.000000,0.639444,-0.287717,0.957716,0.965926,-0.258819,2016,2016-12-15
8653,2016-12-15 08:00:00,3.819917,0.51,28.895042,1.000000,-0.287717,0.957716,0.866025,-0.500000,2016,2016-12-15
8654,2016-12-15 09:00:00,25.095000,0.52,218.696274,1.000000,-0.287717,0.957716,0.707107,-0.707107,2016,2016-12-15
8655,2016-12-15 10:00:00,34.012167,0.55,394.275177,1.000000,-0.287717,0.957716,0.500000,-0.866025,2016,2016-12-15
8656,2016-12-15 11:00:00,38.183783,0.45,535.550598,1.000000,-0.287717,0.957716,0.258819,-0.965926,2016,2016-12-15


In [26]:
# Take all splits
my_splits_all = k_fold_split_option_a(
    df_train=df_train,
    good_ends=good_ends_external,
    n_splits=-1,
    window_size=None,
    gap_day=False,
    front_or_back='front',
    return_type='index'
)

In [27]:
len(my_splits_all)

209